# Benchmarks: reproduce the numbers

MemBukkit ships the full evaluation harness it was developed against. Three long-term-memory benchmarks are wired in:

- **LongMemEval (S)** — 500 questions over long multi-session chat histories: needle recall, temporal reasoning, knowledge updates, abstention.
- **LoCoMo** — very long two-person conversations with QA across sessions.
- **BEAM** — conversations from 100K up to 10M tokens, 20 probing questions each across 9 ability categories.

Every claimed score is backed by a **frozen recipe** that pins the reader, distiller, judge, and encoder to the exact models the number was measured with — `membukkit bench --repro <recipe-id>` reruns it. Every run prints a cost estimate **before** spending anything, and `--lite` runs a small smoke subset first — always start there.

Because LLM readers and judges are nondeterministic, reproduction means landing inside a tolerance band (±0.01 for accuracy metrics, ±0.02 for BEAM averages), not matching bit-for-bit. Full details in [the reproduction guide](../docs/guide/benchmarks.md).

In [ ]:
# The frozen recipes: id, pinned models, expected score, env needs, cost estimate.
!membukkit bench --list

In [ ]:
# Recommended first run: the cheapest recipe on a lite smoke subset (a few haystacks).
# Costs cents, finishes in minutes, and verifies keys + dataset + encoder downloads.
# Lite scores are NOT comparable to the expected numbers — drop --lite for the real thing.
# Needs OPENAI_API_KEY; the dataset downloads from the HF Hub on first use.
!membukkit bench --repro longmemeval-gpt4o-mini --lite

# Full frozen runs (each prints its cost estimate; --yes skips the confirmation prompt).
# gemma recipes additionally need COMPAT_BASE_URL + COMPAT_API_KEY (e.g. DeepInfra).
# !membukkit bench --repro longmemeval-gpt4o-mini --yes    # 82.0%, ~60M tokens (~$9 at 4o-mini rates)
# !membukkit bench --repro longmemeval-gpt54 --yes         # 92.6%, same volume at gpt-5.4 rates
# !membukkit bench --repro longmemeval-gemma --yes         # 88.8%
# !membukkit bench --repro locomo-mem0 --yes               # 87.5%, ~9M tokens
# !membukkit bench --repro beam-100k-gemma --yes           # 0.535, ~2.5M tokens
# !membukkit bench --repro beam-1m-gemma --yes             # 0.498, ~40M tokens
# !membukkit bench --repro beam-10m-gemma --yes            # 0.447, ~120M tokens

# After a full run finishes, verify the score against the tolerance band:
# !membukkit bench --repro longmemeval-gpt4o-mini --check

## Reading the results

Each recipe writes to its own directory, `results/bench/<recipe-id>/`. The headline number lives in `e2e_summary.json` (field `acc`) for LongMemEval and LoCoMo, or `beam_summary.json` (field `average`) for BEAM, alongside per-question verdicts, ability breakdowns, and retrieval traces. The cell below reads whatever recipes have finished and compares each against its expected score.

In [ ]:
import json
from pathlib import Path

# recipe id -> (summary file, headline field, expected score, tolerance)
RECIPES = {
    "longmemeval-gpt54":      ("e2e_summary.json",  "acc",     0.926, 0.01),
    "longmemeval-gemma":      ("e2e_summary.json",  "acc",     0.888, 0.01),
    "longmemeval-gpt4o-mini": ("e2e_summary.json",  "acc",     0.820, 0.01),
    "locomo-mem0":            ("e2e_summary.json",  "acc",     0.875, 0.01),
    "beam-100k-gemma":        ("beam_summary.json", "average", 0.535, 0.02),
    "beam-1m-gemma":          ("beam_summary.json", "average", 0.498, 0.02),
    "beam-10m-gemma":         ("beam_summary.json", "average", 0.447, 0.02),
}

bench_dir = Path("../results/bench")
found = False
for recipe, (summary_file, field, expected, tol) in RECIPES.items():
    path = bench_dir / recipe / summary_file
    if not path.exists():
        continue
    found = True
    score = json.loads(path.read_text())[field]
    status = "within band" if abs(score - expected) <= tol else "OUTSIDE band"
    print(f"{recipe:<26} {field}={score:.3f}  expected {expected:.3f} ±{tol}  -> {status}")

if not found:
    print("no results yet — run a bench cell above first")
    print("(note: lite runs are smoke tests; their scores are not expected to match the band)")

## Tips for full runs

- **Distillation dominates cost — and each recipe pins its own distill cache.** The distiller materially affects scores, so recipes never share caches; rerunning the *same* recipe (after an interruption, or with `--check` in mind) reuses its cache and is nearly free past the first pass.
- **Want a different reader?** Use the free-form presets (`membukkit bench longmemeval --lite --reader ...`): any `openai:` model, Anthropic, Gemini, an OpenAI-compatible host via `COMPAT_BASE_URL`, or a local Ollama model. These are for experiments — only `--repro <recipe-id>` reproduces a claimed number.
- **BEAM scales**: start with `beam-100k-gemma`. The 10M recipe ingests ~10M tokens of conversation per haystack (~120M LLM input tokens total) — check the printed estimate before saying yes.
- The full research flag surface (routing policies, scan budgets, ablations) is available under `membukkit eval --help`.